In [3]:
import sys
sys.path.append('/Users/aidanmorson/Desktop/analysis/parkinsons/analysis')

In [28]:
from analysis.position_sd import get_neuron_positions, compute_average_distance, compute_nearest_neighbor_distances
from analysis.stats_sd import reduce_dimensionality
from load_data.load_npz import load_spikedata
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde


In [5]:
sd24578_49 = load_spikedata('/Users/aidanmorson/Desktop/analysis/parkinsons/data-by-day/d49/24578_D49-6.zip')
sd24578_60 = load_spikedata('/Users/aidanmorson/Desktop/analysis/parkinsons/data-by-day/d60/24578_D60+6.zip')
sd24481_49 = load_spikedata('/Users/aidanmorson/Desktop/analysis/parkinsons/data-by-day/d49/24481_D49-6.zip')
sd24481_60 = load_spikedata('/Users/aidanmorson/Desktop/analysis/parkinsons/data-by-day/d60/24481_D60+6.zip')

In [4]:
positions_578_49 = get_neuron_positions(sd24578_49)
positions_578_60 = get_neuron_positions(sd24578_60)
positions_481_49 = get_neuron_positions(sd24481_49)
positions_481_60 = get_neuron_positions(sd24481_60)

In [5]:
avg_578_49 = compute_average_distance(positions_578_49)
avg_578_60 = compute_average_distance(positions_578_60)
avg_481_49 = compute_average_distance(positions_481_49)
avg_481_60 = compute_average_distance(positions_481_60)

In [6]:
nn_578_49 = compute_nearest_neighbor_distances(positions_578_49)
nn_578_60 = compute_nearest_neighbor_distances(positions_578_60)
nn_481_49 = compute_nearest_neighbor_distances(positions_481_49)
nn_481_60 = compute_nearest_neighbor_distances(positions_481_60)

In [6]:
sdlist24481 = [sd24481_49, sd24481_60]
sdlist24578 = [sd24578_49, sd24578_60]
datasetnames_24481 = [f"24481 D49: {sd24481_49.N} Units ", f"24481 D60: {sd24481_60.N} Units"]
datasetnames_24578 = [f"24578 D49: {sd24578_49.N} Units", f"24578 D60: {sd24578_60.N} Units"]


d49list = [sd24481_49, sd24578_49]
d60list = [sd24481_60, sd24578_60]
datasetnamesd49 = [f"24481: {sd24481_49.N} Units", f"24578: {sd24578_49.N} Units"]
datasetnamesd60 = [f"24481: {sd24481_60} Units", f"24578 {sd24578_60.N} Units"]

full_list = [sd24481_49, sd24481_60, sd24578_49, sd24578_60]
full_names = [f"24481 D49: {sd24481_49.N} Units ", f"24481 D60: {sd24481_60.N} Units", f"24578 D49: {sd24578_49.N} Units", f"24578 D60: {sd24578_60.N} Units"]

s_24481_49_list = [sd24481_49]
sn_name_24481_49 = [f"24481 D49: {sd24481_49.N} Units "]
s_24481_60_list = [sd24481_60]
sn_name_24481_60 = [f"24481 D60: {sd24481_60.N} Units "]
s_24578_49_list = [sd24578_49]
sn_name_24578_49 = [f"24578 D49: {sd24578_49.N} Units"]
s_24578_60_list = [sd24578_60]
sn_name_24578_60 = [f"24578 D60: {sd24578_60.N} Units"]



In [10]:
full_colors = ["green", "lime", "royalblue", "cyan"]
colors24481 = ["green", "lime"]
colors24578 = ["royalblue", "cyan"]
cs_color_481_49 = ["green"]
cs_color_481_60 = ["lime"]
cs_color_578_49 = ["royalblue"]
cs_color_578_60 = ["cyan"]

In [ ]:
def plot_avg_distance_histogram_overlaid(
    sd_list,
    dataset_names,
    bins_range=(0, 1000),
    n_bins=50,
    output_path=None
):
    """
    Compute and plot overlaid average-distance histograms for multiple SpikeData objects
    on a single figure.

    For each SpikeData in sd_list:
      1. Retrieve neuron positions via get_neuron_positions(sd).
      2. Compute the pairwise distance matrix and the average distance per neuron.
      3. Build a histogram (counts vs. distance bin centers).
    Then overlay these histograms on a single plot.

    Parameters:
    -----------
    sd_list : list of SpikeData
        A list of SpikeData objects (e.g., from the same organoid but different days).
    dataset_names : list of str
        Names/labels for each dataset (same length as sd_list).
    bins_range : tuple of (float, float), default (0, 1000)
        The min and max for the x-axis in µm.
    n_bins : int, default 50
        Number of bins for the histogram.
    output_path : str, optional
        If provided, the figure is saved to this path (PNG). Otherwise, plotted interactively.
        e.g., "/path/to/avg_distance_overlay.png"
    figsize : tuple, default (8, 6)
        Figure size in inches.
    """
    # A small color palette for multiple datasets
    colors = ["green", "lime", "royalblue", "cyan"]

    plt.figure(figsize=(10, 8))

    for i, sd in enumerate(sd_list):
        dataset_name = dataset_names[i] if i < len(dataset_names) else f"Dataset_{i+1}"
        c = colors[i % len(colors)]
        
        # 1. Get positions using your get_neuron_positions function.
        full_positions = get_neuron_positions(sd)
        valid_positions = []
        # Filter out invalid [0, 0] positions if needed.
        for pos in full_positions:
            if not np.allclose(pos, [0, 0]):
                valid_positions.append(pos)
        # Convert list to a NumPy array.
        valid_positions = np.array(valid_positions)
        
        if valid_positions.size == 0:
            print(f"No valid positions for dataset {dataset_name}, skipping.")
            continue
        
        # 2. Compute pairwise distances.
        avg_distance = compute_average_distance(valid_positions)

        # 3. Build histogram.
        bin_edges = np.linspace(bins_range[0], bins_range[1], n_bins + 1)
        counts, _ = np.histogram(avg_distance, bins=bin_edges)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        # Overlay the histogram as a line plot.
        #plt.plot(bin_centers, counts, marker='o', linestyle='-', color=c, label=dataset_name) #meh

        #use hist instead it looks better
        plt.hist(avg_distance, bins=bin_edges, color=c, alpha=.9, label=dataset_name)
    
    plt.xlabel("Average Distance to all Neurons(µm)")
    plt.ylabel("Number of Neurons")
    plt.title("Average Distance Per Neuron")
    plt.xlim(bins_range)
    plt.legend()
    plt.tight_layout()

    if output_path:
        # If output_path is a directory, save with a default filename.
        if os.path.isdir(output_path):
            save_path = os.path.join(output_path, "avg_distance_histogram_overlaid.png")
        else:
            save_path = output_path
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()



In [ ]:
plot_avg_distance_histogram_overlaid(sdlist24578, datasetnames_24578, bins_range=(200, 1000), n_bins=100)

In [ ]:
def plot_avg_distance_smoothed_histogram_overlaid(
    sd_list,
    dataset_names,
    colors,
    bins_range=(0, 1000),
    n_bins=50,
    output_path=None,
    filename = None
):
    """
    Compute and plot overlaid average-distance histograms for multiple SpikeData objects
    on a single figure.

    For each SpikeData in sd_list:
      1. Retrieve neuron positions via get_neuron_positions(sd).
      2. Compute the pairwise distance matrix and the average distance per neuron.
      3. Build a histogram (counts vs. distance bin centers).
    Then overlay these histograms on a single plot.

    Parameters:
    -----------
    sd_list : list of SpikeData
        A list of SpikeData objects (e.g., from the same organoid but different days).
    dataset_names : list of str
        Names/labels for each dataset (same length as sd_list).
    colors : list of str
        Colors for each dataset (same length as sd_list).
    bins_range : tuple of (float, float), default (0, 1000)
        The min and max for the x-axis in µm.
    n_bins : int, default 50
        Number of bins for the histogram.
    output_path : str, optional
        If provided, the figure is saved to this path (PNG). Otherwise, plotted interactively.
        e.g., "/path/to/avg_distance_overlay.png"
    filename : str, optional
        If provided, the figure is saved with this filename (PNG). Otherwise, a default filename is used.

    """
    # A small color palette for multiple datasets

    plt.figure(figsize=(10, 8))

    x_vals = np.linspace(bins_range[0], bins_range[1], n_bins)
    dx = x_vals[1] - x_vals[0]
    bar_width = dx * .8

    for i, sd in enumerate(sd_list):
        dataset_name = dataset_names[i] if i < len(dataset_names) else f"Dataset_{i+1}"
        c = colors[i % len(colors)]
        
        # 1. Get positions using your get_neuron_positions function.
        full_positions = get_neuron_positions(sd)
        valid_positions = []
        # Filter out invalid [0, 0] positions if needed.
        for pos in full_positions:
            if not np.allclose(pos, [0, 0]):
                valid_positions.append(pos)
        # Convert list to a NumPy array.
        valid_positions = np.array(valid_positions)
        
        if valid_positions.size == 0:
            print(f"No valid positions for dataset {dataset_name}, skipping.")
            continue
        
        # 2. Compute pairwise distances.
        avg_distance = compute_average_distance(valid_positions)

        # 3. smooth
        kde = gaussian_kde(avg_distance)
        density = kde(x_vals)

        # Overlay the histogram as a line plot.
        #plt.plot(bin_centers, counts, marker='o', linestyle='-', color=c, label=dataset_name) #meh

        #use hist instead it looks better
        plt.bar(x_vals, density, width = bar_width, color=c, alpha=.6, label=dataset_name)
    
    plt.xlabel("Average Distance to all Neurons(µm)")
    plt.ylabel("Probability Density")
    plt.title("Average Distance Per Neuron")
    plt.xlim(bins_range)
    plt.legend()
    plt.tight_layout()

    if filename != None and output_path == None:
        print("You passed a filename but no output path, please provide an output path")

    if output_path:
        # If output_path is a directory, save with a default filename.
        if os.path.isdir(output_path):
            save_path = os.path.join(output_path, f"avg_distance_hist_smooth_{filename}.png")
        else:
            save_path = output_path
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()



In [ ]:
plot_avg_distance_smoothed_histogram_overlaid(sdlist24481, datasetnames_24481, colors24481, bins_range=(200, 1000), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/avg/", filename="24481")
plot_avg_distance_smoothed_histogram_overlaid(sdlist24578, datasetnames_24578, colors24578, bins_range=(200, 1000), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/avg/", filename="24578")
plot_avg_distance_smoothed_histogram_overlaid(full_list, full_names, full_colors, bins_range=(200, 1000), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/avg/", filename="full")

In [ ]:
plot_avg_distance_smoothed_histogram_overlaid(s_24481_49_list, sn_name_24481_49, cs_color_481_49, bins_range=(200, 1000), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/avg/", filename="24481_d49")
plot_avg_distance_smoothed_histogram_overlaid(s_24481_60_list, sn_name_24481_60, cs_color_481_60, bins_range=(200, 1000), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/avg/", filename="24481_d60")
plot_avg_distance_smoothed_histogram_overlaid(s_24578_49_list, sn_name_24578_49, cs_color_578_49, bins_range=(200, 1000), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/avg/", filename="24578_d49")
plot_avg_distance_smoothed_histogram_overlaid(s_24578_60_list, sn_name_24578_60, cs_color_578_60, bins_range=(200, 1000), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/avg/", filename="24578_d60")

In [ ]:
def plot_nearest_neighbor_histogram_overlaid(
    sd_list,
    dataset_names,
    colors,
    bins_range=(0, 1000),
    n_bins=50,
    output_path=None,
    filename=None
):
    """
    Compute and plot overlaid nearest-neighbor histograms for multiple SpikeData objects
    on a single figure.

    For each SpikeData in sd_list:
      1. Retrieve neuron positions via get_neuron_positions(sd).
      2. Retrieve nearest neighobors via nearest_neighbor_distances(positions).
      3. Build a histogram (counts vs. distance bin centers).
    Then overlay these histograms on a single plot.

    Parameters:
    -----------
    sd_list : list of SpikeData
        A list of SpikeData objects (e.g., from the same organoid but different days).
    dataset_names : list of str
        Names/labels for each dataset (same length as sd_list).
    bins_range : tuple of (float, float), default (0, 1000)
        The min and max for the x-axis in µm.
    n_bins : int, default 50
        Number of bins for the histogram.
    colors : list of color
        A list of colors corresponding exactly for continuty across plots.
    output_path : str, optional
        If provided, the figure is saved to this path (PNG). Otherwise, plotted interactively.
        e.g., "/path/to/avg_distance_overlay.png"
    figsize : tuple, default (8, 6)
        Figure size in inches.
    Filename : str, optional
        If provided, the figure is saved with this filename attached to the end.
    """
    # A small color palette for multiple datasets

    plt.figure(figsize=(10, 8))

    for i, sd in enumerate(sd_list):
        dataset_name = dataset_names[i] if i < len(dataset_names) else f"Dataset_{i+1}"
        c = colors[i % len(colors)]
        
        # 1. Get positions using your get_neuron_positions function.
        full_positions = get_neuron_positions(sd)
        valid_positions = []
        # Filter out invalid [0, 0] positions if needed.
        for pos in full_positions:
            if not np.allclose(pos, [0, 0]):
                valid_positions.append(pos)
        # Convert list to a NumPy array.
        valid_positions = np.array(valid_positions)
        
        if valid_positions.size == 0:
            print(f"No valid positions for dataset {dataset_name}, skipping.")
            continue
        
        # 2. Compute pairwise distances.
        nearest_neighbor = compute_nearest_neighbor_distances(valid_positions)

        # 3. Build histogram.
        bin_edges = np.linspace(bins_range[0], bins_range[1], n_bins + 1)
        counts, _ = np.histogram(nearest_neighbor, bins=bin_edges)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        # Overlay the histogram as a line plot.
        #plt.plot(bin_centers, counts, marker='o', linestyle='-', color=c, label=dataset_name) #meh
        #hist is better
        plt.hist(nearest_neighbor, bins=bin_edges, color=c, alpha=.9, label=dataset_name)
    
    plt.xlabel("Nearest Neighbor Distance(µm)")
    plt.ylabel("Number of Neurons")
    plt.title("Distance to Nearest Neighbor Per Neuron")
    plt.xlim(bins_range)
    plt.legend()
    plt.tight_layout()

    if filename != None and output_path == None:
        print("You passed a filename but no output path, please provide an output path")

    if output_path:
        # If output_path is a directory, save with a default filename.
        if os.path.isdir(output_path):
            save_path = os.path.join(output_path, f"nn-hist-{filename}.png")
        else:
            save_path = output_path
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()


In [ ]:
def plot_nearest_neighbor_smoothed(
    sd_list,
    dataset_names,
    colors,
    bins_range=(0, 1000),
    n_bins=50,
    output_path=None,
    filename=None
):
    """
    Compute and plot overlaid nearest-neighbor histograms for multiple SpikeData objects
    on a single figure.

    For each SpikeData in sd_list:
      1. Retrieve neuron positions via get_neuron_positions(sd).
      2. Retrieve nearest neighobors via nearest_neighbor_distances(positions).
      3. Build a histogram (counts vs. distance bin centers).
    Then overlay these histograms on a single plot.

    Parameters:
    -----------
    sd_list : list of SpikeData
        A list of SpikeData objects (e.g., from the same organoid but different days).
    dataset_names : list of str
        Names/labels for each dataset (same length as sd_list).
    bins_range : tuple of (float, float), default (0, 1000)
        The min and max for the x-axis in µm.
    n_bins : int, default 50
        Number of bins for the histogram.
    colors : list of color
        A list of colors corresponding exactly for continuty across plots.
    output_path : str, optional
        If provided, the figure is saved to this path (PNG). Otherwise, plotted interactively.
        e.g., "/path/to/avg_distance_overlay.png"
    figsize : tuple, default (8, 6)
        Figure size in inches.
    Filename : str, optional
        If provided, the figure is saved with this filename attached to the end.
    """
    # A small color palette for multiple datasets

    plt.figure(figsize=(10, 8))

    x_vals = np.linspace(bins_range[0], bins_range[1], n_bins)
    dx = x_vals[1] - x_vals[0]
    bar_width = dx * .8

    for i, sd in enumerate(sd_list):
        dataset_name = dataset_names[i] if i < len(dataset_names) else f"Dataset_{i+1}"
        c = colors[i % len(colors)]
        
        # 1. Get positions using your get_neuron_positions function.
        full_positions = get_neuron_positions(sd)
        valid_positions = []
        # Filter out invalid [0, 0] positions if needed.
        for pos in full_positions:
            if not np.allclose(pos, [0, 0]):
                valid_positions.append(pos)
        # Convert list to a NumPy array.
        valid_positions = np.array(valid_positions)
        
        if valid_positions.size == 0:
            print(f"No valid positions for dataset {dataset_name}, skipping.")
            continue
        
        # 2. Compute pairwise distances.
        nearest_neighbor = compute_nearest_neighbor_distances(valid_positions)

        #3. smooth
        kde = gaussian_kde(nearest_neighbor)
        density = kde(x_vals)

        # Overlay the histogram as a line plot.
        #plt.plot(bin_centers, counts, marker='o', linestyle='-', color=c, label=dataset_name) #meh
        #hist is better
        plt.bar(x_vals, density, width=bar_width, color=c, alpha=.6, label=dataset_name)
    
    plt.xlabel("Nearest Neighbor Distance(µm)")
    plt.ylabel("Probability Density")
    plt.title("Distance to Nearest Neighbor Per Neuron")
    plt.xlim(bins_range)
    plt.legend()
    plt.tight_layout()

    if filename != None and output_path == None:
        print("You passed a filename but no output path, please provide an output path")

    if output_path:
        # If output_path is a directory, save with a default filename.
        if os.path.isdir(output_path):
            save_path = os.path.join(output_path, f"nn-smoothed-{filename}.png")
        else:
            save_path = output_path
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()


In [33]:
plot_nearest_neighbor_smoothed(sdlist24481, datasetnames_24481, colors24481, bins_range=(0, 300), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24481")
plot_nearest_neighbor_smoothed(sdlist24578, datasetnames_24578, colors24578, bins_range=(0, 300), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24578")
plot_nearest_neighbor_smoothed(full_list, full_names, full_colors, bins_range=(0, 300), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="full")
plot_nearest_neighbor_smoothed(s_24481_49_list, sn_name_24481_49, cs_color_481_49, bins_range=(0, 300), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24481_d49")
plot_nearest_neighbor_smoothed(s_24481_60_list, sn_name_24481_60, cs_color_481_60, bins_range=(0, 300), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24481_d60")
plot_nearest_neighbor_smoothed(s_24578_49_list, sn_name_24578_49, cs_color_578_49, bins_range=(0, 300), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24578_d49")
plot_nearest_neighbor_smoothed(s_24578_60_list, sn_name_24578_60, cs_color_578_60, bins_range=(0, 300), n_bins=100, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24578_d60")

In [34]:
plot_nearest_neighbor_histogram_overlaid(s_24578_49_list, sn_name_24578_49, bins_range=(0, 300), n_bins=100, colors=cs_color_578_49, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24578-49")
plot_nearest_neighbor_histogram_overlaid(s_24578_60_list, sn_name_24578_60, bins_range=(0, 300), n_bins=100, colors=cs_color_578_60, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24578-60")
plot_nearest_neighbor_histogram_overlaid(s_24481_49_list, sn_name_24481_49, bins_range=(0, 300), n_bins=100, colors=cs_color_481_49, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24481-49")
plot_nearest_neighbor_histogram_overlaid(s_24481_60_list, sn_name_24481_60, bins_range=(0, 300), n_bins=100, colors = cs_color_481_60, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24481-60")
plot_nearest_neighbor_histogram_overlaid(sdlist24481, datasetnames_24481, bins_range=(0, 300), n_bins=100, colors=colors24481, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24481")
plot_nearest_neighbor_histogram_overlaid(sdlist24578, datasetnames_24578, bins_range=(0, 300), n_bins=100, colors=colors24578, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="24578")
plot_nearest_neighbor_histogram_overlaid(full_list, full_names, bins_range=(0, 300), n_bins=100, colors=full_colors, output_path="/Users/aidanmorson/Desktop/analysis/parkinsons/seperation/nn/", filename="full")